# Strategy

In [ ]:
class Order {}

class OrderProcessor
{
    decimal calculateFedExCost(Order order) => 5;
    decimal calculateUpsCost(Order order) => 4;
    decimal calculateUpspCost(Order order) => 3;
    decimal calculateDmlCost(Order order) => 2;

    public decimal CalculateShippingCost(Order order, string shippingProvider)
    {
        switch(shippingProvider)
        {
            case "FedEx":
                return calculateFedExCost(order);
            case "UPS":
                return calculateUpsCost(order);
            case "USPS":
                return calculateUpspCost(order);
            case "DML":
                return calculateDmlCost(order);
            default:
                throw new ArgumentException("Unknown shipping provider", nameof(shippingProvider));
        }
    }
}

- Every new shipping partner requires opening this class and adding a new case
- This violates the Open/Closed Principle

In [ ]:
interface IShippingStrategy
{
    string ProviderName { get; }
    decimal CalculateCost(Order order);
}

class FedExShippingStrategy : IShippingStrategy
{
    public string ProviderName => "FedEx";

    public decimal CalculateCost(Order order) => 5;
}

In [ ]:
class OrderProcessor
{
    readonly Dictionary<string, IShippingStrategy> shippingStrategies;

    public OrderProcessor(IEnumerable<IShippingStrategy> shippingStrategies)
        => (this.shippingStrategies) = (shippingStrategies.ToDictionary(s => s.ProviderName, s => s));

    public decimal CalculateShippingCost(Order order, string shippingProvider)
    {
        if (shippingStrategies.TryGetValue(shippingProvider, out var shippingStrategy))
            return shippingStrategy.CalculateCost(order);
        else
            throw new ArgumentException("Unknown shipping provider", nameof(shippingProvider));
    }
}

Credit: https://www.youtube.com/watch?v=CGRGun-Bw40 Replace Switch Statements with the Strategy Pattern in C#

In [ ]:
var x = new MyClass("strategy pattern", s => s.ToUpper());
x.DoSomething();

var y = new MyClass("strategy pattern", s =>
{
    if (string.IsNullOrEmpty(s))
        return s;

    return char.ToUpper(s[0]) + s[1..];
});
y.DoSomethingSmart();

class MyClass(string value, Func<string, string> strategy)
{
    public void DoSomething()
        => Console.WriteLine(strategy(value));

    public void DoSomethingSmart()
    {
        var result = string.Join(" ",
            value.Split(' ', StringSplitOptions.RemoveEmptyEntries)
             .Select(strategy));

        Console.WriteLine(result);
    }
}

# Pipeline Pattern

In [ ]:
class LoanApplication
{
    public int ApplicationId { get; set; }
    public DateTime DateOfBirth { get; set; }
}


In [ ]:
class EligibilityContext
{
    public required LoanApplication Application { get; init; }
    public List<string> Warnings { get; set; } = [];
    public bool IsRejected { get; set; } = false;
    public string RejectionReason { get; set; } = null;

    public void Reject(string reason)
        => (IsRejected, RejectionReason) = (true, reason);
    
    public void Warn(string reason)
        => Warnings.Add(reason);
}

interface IEligibilityRule
{
    int Order { get; }
    void Evaluate(EligibilityContext context);
}

In [ ]:
class AgeRule : IEligibilityRule
{
    public int Order => 10;

    public void Evaluate(EligibilityContext context)
    {
        int age = DateTime.Today.Year - context.Application.DateOfBirth.Year;
        if (context.Application.DateOfBirth > DateTime.Today.AddYears(-age)) age--;

        if (age < 18)
            context.Reject("Applicant must be at least 18 years old");
        if (age > 65)
            context.Warn("Applicant age exceeds standard policy limit");
    }
}

In [ ]:
enum LoanStatus { Approved, ApprovedWithConditions, ManualReview, Rejected }

class LoanDecision
{
    public int ApplicationId { get; set; }
    public LoanStatus Status { get; set; }
    public IEnumerable<string> Reasons { get; set; }
}

class ElibibilityPipeline
{
    readonly IEnumerable<IEligibilityRule> rules;

    public ElibibilityPipeline(IEnumerable<IEligibilityRule> rules) // DI can fill them in
        => (this.rules) = rules.OrderBy(r => r.Order);

    public LoanDecision Run(LoanApplication loanApplication)
    {
        var context = new EligibilityContext
        {
            Application = loanApplication
        };

        foreach(var rule in rules)
        {
            rule.Evaluate(context);

            if (context.IsRejected)
                return new LoanDecision
                {
                    ApplicationId = loanApplication.ApplicationId,
                    Status = LoanStatus.Rejected,
                    Reasons = [context.RejectionReason ?? "N/A"]
                };
        }

        var status = context.Warnings.Count switch
        {
            0 => LoanStatus.Approved,
            <= 2 => LoanStatus.ApprovedWithConditions,
            _ => LoanStatus.ManualReview
        };

        return new LoanDecision
        {
            ApplicationId = loanApplication.ApplicationId,
            Status = status,
            Reasons = context.Warnings
        };
    }
}

Credit: https://www.youtube.com/watch?v=RfknMfzTUbo Refactoring a 500-Line Method with the Pipeline Pattern